# 14. Sesgo de intensidad de AIFS 2023–2024 (seguimiento D24, emparejamiento en t0)

Figura del problema para la corrección tipo AIFS-TC (prueba piloto, reunión con Tom del 7 dic).
Entrada: `matches_long.csv` y `misses.csv` de `scripts/tracking/match_tracks_ibtracs.py` (AITCHazard), copiados desde
`$BASE/outputs/tracking/match_d24/`. Viento de AIFS dividido entre 0.88 (convención de TCBench); viento observado
`USA_WIND` (1 min). Solo emparejamientos en t0 y registros de IBTrACS con NATURE = TS en el tiempo válido.

Intervalos de confianza: bootstrap por tormenta (se remuestrean tormentas completas), porque los inicios
consecutivos de una misma tormenta comparten el error y no son independientes.

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from itchi.plotting import MapStyle, configure_map_axis, get_crameri_colormap

DATA = Path("/mnt/d/AITCHazard/tracking/match_d24")   # copia local de $BASE/outputs/tracking/match_d24
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:  # itchi-core root, wherever Jupyter starts
    ROOT = ROOT.parent
FIGS = ROOT / "outputs/figures/aitchazard"
FIGS.mkdir(parents=True, exist_ok=True)
LANG = "en"                                          # "en" para Tom, "es" para el comité
N_BOOT = 2000
RNG = np.random.default_rng(0)
LEAD_B = 48                                          # plazo comparado con t0 en los paneles d-f

STYLE = MapStyle(lon_extent=(-135.0, -55.0), lat_extent=(0.0, 40.0),
                 lon_tick_spacing=20.0, lat_tick_spacing=10.0, grid_spacing=10.0)
CMAP = get_crameri_colormap("batlow")
CMAP_DIV = get_crameri_colormap("vik")
C0, C1 = CMAP(0.15), CMAP(0.65)
plt.rcParams.update({"font.size": STYLE.label_size, "axes.linewidth": STYLE.spine_linewidth,
                     "xtick.major.width": STYLE.tick_width, "ytick.major.width": STYLE.tick_width,
                     "xtick.major.size": STYLE.tick_size, "ytick.major.size": STYLE.tick_size})

T = {"en": dict(lead="Lead time (h)", vbias="$V_{max}$ error (kt)", pbias="$P_{min}$ error (hPa)",
                bias="bias", mae="MAE", drift="drift (error $-$ error at 0 h)", obs_cat="Observed category",
                fc_cat="Forecast $V_{max}$ (kt)", e0="$V_{max}$ error at 0 h (kt)",
                eL=f"$V_{{max}}$ error at {LEAD_B} h (kt)", map="$V_{max}$ error at 0 h (kt)"),
     "es": dict(lead="Plazo (h)", vbias="Error de $V_{max}$ (kt)", pbias="Error de $P_{min}$ (hPa)",
                bias="sesgo", mae="EAM", drift="deriva (error $-$ error en 0 h)", obs_cat="Categoría observada",
                fc_cat="$V_{max}$ pronosticado (kt)", e0="Error de $V_{max}$ en 0 h (kt)",
                eL=f"Error de $V_{{max}}$ a {LEAD_B} h (kt)", map="Error de $V_{max}$ en 0 h (kt)")}[LANG]

In [ ]:
long = pd.read_csv(DATA / "matches_long.csv", parse_dates=["init", "valid"])
misses = pd.read_csv(DATA / "misses.csv", parse_dates=["init"])
d = long[(long["match"] == "t0") & long["ob_nature"].eq("TS")
         & long["fc_vmax_kt"].notna() & long["ob_vmax_kt"].notna()].copy()

SS_EDGES = [0, 34, 64, 83, 96, 113, 137, 400]
SS_LABELS = ["TD", "TS", "H1", "H2", "H3", "H4", "H5"] if LANG == "en" else ["DT", "TT", "H1", "H2", "H3", "H4", "H5"]
FC_EDGES = [0, 34, 50, 64, 400]
FC_LABELS = ["<34", "34–50", "50–64", "≥64"]
d["cat_ob"] = pd.cut(d["ob_vmax_kt"], SS_EDGES, right=False, labels=SS_LABELS)
d["cat_fc"] = pd.cut(d["fc_vmax_kt"], FC_EDGES, right=False, labels=FC_LABELS)

# error at 0 h of the same forecast track: how much of the error at lead L is inherited from t0
key = ["init", "SID", "track"]
e0 = d[d["lead_h"] == 0].set_index(key)["err_vmax_kt"].rename("e0")
d = d.join(e0, on=key)
d["drift"] = d["err_vmax_kt"] - d["e0"]
print(f"{len(d)} pairs, {d['SID'].nunique()} storms, {d.drop_duplicates(['init', 'SID']).shape[0]} storm-inits")

In [ ]:
def storm_boot(df, col, n=N_BOOT, q=(0.05, 0.95)):
    """Mean of col and 90 % CI resampling whole storms."""
    g = df.dropna(subset=[col]).groupby("SID")[col].agg(["sum", "count"])
    if len(g) < 2:
        return np.nan, np.nan, np.nan, len(g)
    s, c = g["sum"].to_numpy(), g["count"].to_numpy()
    idx = RNG.integers(0, len(g), size=(n, len(g)))
    boots = s[idx].sum(1) / c[idx].sum(1)
    return s.sum() / c.sum(), *np.quantile(boots, q), len(g)


rows = []
for L, g in d.groupby("lead_h"):
    vm, vl, vh, ns = storm_boot(g, "err_vmax_kt")
    pm, pl, ph, _ = storm_boot(g, "err_pres_hPa")
    dm, dl, dh, _ = storm_boot(g, "drift")
    rows.append(dict(lead_h=L, n=len(g), n_storms=ns, vmax_bias=vm, vmax_lo=vl, vmax_hi=vh,
                     vmax_mae=g["err_vmax_kt"].abs().mean(), drift=dm, drift_lo=dl, drift_hi=dh,
                     pres_bias=pm, pres_lo=pl, pres_hi=ph,
                     r_e0=g[["err_vmax_kt", "e0"]].corr().iloc[0, 1]))
by_lead = pd.DataFrame(rows).set_index("lead_h")
by_lead.round(2)

In [ ]:
def by_cat(df, cat):
    out = []
    for L in (0, LEAD_B):
        for c, g in df[df["lead_h"] == L].groupby(cat, observed=False):
            m, lo, hi, ns = storm_boot(g, "err_vmax_kt")
            out.append(dict(lead_h=L, cat=c, n=len(g), n_storms=ns, bias=m, lo=lo, hi=hi))
    return pd.DataFrame(out)


cat_ob, cat_fc = by_cat(d, "cat_ob"), by_cat(d, "cat_fc")
display(cat_ob.round(1)), display(cat_fc.round(1))

In [ ]:
# detection at t0 by observed intensity (NATURE TS): hits from the 0 h rows, misses from misses.csv
hits0 = d[d["lead_h"] == 0].drop_duplicates(["init", "SID"])[["ob_vmax_kt"]].assign(hit=1)
miss0 = misses[misses["nature"].eq("TS")][["vmax_kt"]].rename(columns={"vmax_kt": "ob_vmax_kt"}).assign(hit=0)
det = pd.concat([hits0, miss0])
det["cat"] = pd.cut(det["ob_vmax_kt"], SS_EDGES, right=False, labels=SS_LABELS)
pod = det.groupby("cat", observed=False)["hit"].agg(n="size", pod="mean")
pod.round(2)

**Lectura.** El panel b separa el error medio en lo heredado del estado inicial (ERA5 en t0) y la deriva del
modelo con el plazo. Los paneles d y e condicionan el mismo error de dos formas: por la intensidad observada,
que exagera el sesgo de las tormentas intensas por regresión a la media, y por la intensidad pronosticada, que es
lo único que se conoce al pronosticar y es el condicionamiento que usa la corrección.

In [ ]:
fig = plt.figure(figsize=(16, 9.5))
gs = fig.add_gridspec(2, 3, hspace=0.32, wspace=0.38)
crs = ccrs.PlateCarree()

# (a) positions at t0 coloured by the initial Vmax error
ax = fig.add_subplot(gs[0, 0], projection=crs)
configure_map_axis(ax, crs=crs, style=STYLE)
z = d[d["lead_h"] == 0]
sc = ax.scatter(z["ob_lon"], z["ob_lat"], c=z["err_vmax_kt"], cmap=CMAP_DIV, vmin=-80, vmax=80,
                s=14, edgecolor="k", linewidth=0.2, transform=crs, zorder=5)
fig.colorbar(sc, ax=ax, orientation="horizontal", pad=0.12, shrink=0.9, label=T["map"])

# (b) Vmax bias, MAE and drift by lead
ax = fig.add_subplot(gs[0, 1])
x = by_lead.index
ax.fill_between(x, by_lead["vmax_lo"], by_lead["vmax_hi"], color=C0, alpha=0.25, lw=0)
ax.plot(x, by_lead["vmax_bias"], "-o", color=C0, ms=4, label=T["bias"])
ax.plot(x, -by_lead["vmax_mae"], "--", color=C0, label=f"$-${T['mae']}")
ax.fill_between(x, by_lead["drift_lo"], by_lead["drift_hi"], color=C1, alpha=0.25, lw=0)
ax.plot(x, by_lead["drift"], "-s", color=C1, ms=4, label=T["drift"])
ax.axhline(0, color="0.3", lw=0.8)
ax.set(xlabel=T["lead"], ylabel=T["vbias"], xticks=range(0, 73, 12))
ax.legend(frameon=False, fontsize=9)
ax2 = ax.twinx()
ax2.bar(x, by_lead["n"], width=4, color="0.85", zorder=0)
ax2.set_ylabel("n", color="0.5")
ax2.tick_params(axis="y", colors="0.5")
ax.set_zorder(ax2.get_zorder() + 1)
ax.patch.set_visible(False)

# (c) Pmin bias by lead
ax = fig.add_subplot(gs[0, 2])
ax.fill_between(x, by_lead["pres_lo"], by_lead["pres_hi"], color=C0, alpha=0.25, lw=0)
ax.plot(x, by_lead["pres_bias"], "-o", color=C0, ms=4)
ax.axhline(0, color="0.3", lw=0.8)
ax.set(xlabel=T["lead"], ylabel=T["pbias"], xticks=range(0, 73, 12))


def cat_panel(ax, tab, labels, xlabel):
    xi = np.arange(len(labels))
    for k, (L, col) in enumerate(((0, C0), (LEAD_B, C1))):
        t = tab[tab["lead_h"] == L].set_index("cat").reindex(labels)
        ok = t["n_storms"] >= 2
        ax.errorbar(xi[ok] + (k - 0.5) * 0.25, t["bias"][ok],
                    yerr=[(t["bias"] - t["lo"])[ok], (t["hi"] - t["bias"])[ok]],
                    fmt="o", color=col, capsize=3, label=f"{L} h")
        for xx, (_, r) in zip(xi, t.iterrows()):
            if k == 0 and np.isfinite(r["n"]):
                ax.annotate(f"{int(r['n'])}", (xx, 1.0), xycoords=("data", "axes fraction"),
                            ha="center", va="bottom", fontsize=8, color="0.4")
    ax.axhline(0, color="0.3", lw=0.8)
    ax.set_xticks(xi, labels)
    ax.set(xlabel=xlabel, ylabel=T["vbias"])
    ax.legend(frameon=False, fontsize=9)


cat_panel(fig.add_subplot(gs[1, 0]), cat_ob, SS_LABELS, T["obs_cat"])   # (d)
cat_panel(fig.add_subplot(gs[1, 1]), cat_fc, FC_LABELS, T["fc_cat"])    # (e)

# (f) error at LEAD_B against error at 0 h for the same track
ax = fig.add_subplot(gs[1, 2])
z = d[d["lead_h"] == LEAD_B]
ax.scatter(z["e0"], z["err_vmax_kt"], s=10, color=C0, alpha=0.6, edgecolor="none")
lim = [min(z["e0"].min(), z["err_vmax_kt"].min()) - 5, max(z["e0"].max(), z["err_vmax_kt"].max()) + 5]
ax.plot(lim, lim, color="0.3", lw=0.8)
ax.set(xlim=lim, ylim=lim, xlabel=T["e0"], ylabel=T["eL"])
ax.text(0.04, 0.94, f"r = {z[['e0', 'err_vmax_kt']].corr().iloc[0, 1]:.2f}, n = {len(z)}",
        transform=ax.transAxes, va="top")

for letter, a in zip("abcdef", [a for a in fig.axes if a.get_label() != "<colorbar>" and a.get_ylabel() != "n"]):
    a.text(-0.02, 1.04, f"({letter})", transform=a.transAxes, fontweight="bold", ha="right")

out = FIGS / f"aifs_intensity_bias_2023_2024_{LANG}"
fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out.with_suffix(".png"), dpi=300, bbox_inches="tight")
print("saved", out.with_suffix(".pdf"))